In [11]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf transformers sentence-transformers accelerate

In [12]:
from google.colab import files

uploaded = files.upload()

Saving Chapter 7.pdf to Chapter 7 (1).pdf


In [13]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Chapter 7 (1).pdf")
documents = loader.load()

print("Number of pages:", len(documents))
print("\nFirst 500 characters:\n")
print(documents[0].page_content[:500])

Number of pages: 7

First 500 characters:

7. Explain Various types of Vulnerabilities 
Application Vulnerabilities 
Application vulnerabilities are flaws or weaknesses in software that malicious actors can 
exploit to compromise systems. These include: 
1. Memory Injection 
• Definition: Secret insertion of malicious code into a program’s memory. 
• Impact: Unauthorized access, arbitrary code execution, data breaches. 
• Tools: Forensic toolkits may exploit memory injection to interact with running 
applications. 
• Detection Challenge 


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))
print("\nFirst Chunk:\n")
print(docs[0].page_content)

Total Chunks: 29

First Chunk:

7. Explain Various types of Vulnerabilities 
Application Vulnerabilities 
Application vulnerabilities are flaws or weaknesses in software that malicious actors can 
exploit to compromise systems. These include: 
1. Memory Injection 
• Definition: Secret insertion of malicious code into a program’s memory. 
• Impact: Unauthorized access, arbitrary code execution, data breaches. 
• Tools: Forensic toolkits may exploit memory injection to interact with running 
applications.


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded successfully!


In [16]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

print("FAISS Vector Database created successfully!")


FAISS Vector Database created successfully!


In [17]:
query = "What is SQL Injection?"

results = vectorstore.similarity_search(query, k=2)

print("Retrieved Chunks:\n")

for i, doc in enumerate(results):
    print(f"\nChunk {i+1}:")
    print(doc.page_content)

Retrieved Chunks:


Chunk 1:
o Digital signatures 
o Verified update sources 
o Multi Factor Authentication for updates 
• Example: CCleaner supply chain attack  (2017) – Hackers injected malware into a 
legitimate update, affecting millions. 
Web-Based Vulnerabilities 
1. SQL Injection (SQLI) 
• Definition: Malicious SQL code is inserted into input fields to manipulate backend 
databases. 
• Impact: Unauthorized access, data theft, administrative control. 
• Mitigation: 
o Input sanitization and validation

Chunk 2:
o Input sanitization and validation 
o Parameterized queries 
o Stored procedures 
• Example Code Vulnerability: 
user_input = request.GET['product_name'] 
query = "SELECT * FROM products WHERE name = '" + user_input 
+’;" 
result = execute_query(query) 
Once the attacker launches the code above, they notice that the website is vulnerable to 
SQL injection and decides to exploit it.


In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

context = "\n".join([doc.page_content for doc in results])

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", query)
print("\nAnswer:")
print(answer)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question: What is SQL Injection?

Answer:
Malicious SQL code is inserted into input fields to manipulate backend databases
